In [163]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, callback
from jupyter_dash import JupyterDash
import kagglehub
import dash_bootstrap_components as dbc

In [164]:
path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")
files = ['drivers.csv', 'races.csv', 'results.csv', 'circuits.csv', 'constructors.csv', 'qualifying.csv']
drivers, races, results, circuits, constructors, qualifying = [
    pd.read_csv(os.path.join(path, f)) for f in files
]

In [165]:
drivers.drop_duplicates(subset='driverId', inplace=True)
constructors.rename(columns={'name': 'constructor'}, inplace=True)

In [166]:
f1 = results.merge(
    races[['raceId','year','name','date','circuitId']], on='raceId', how='left'
).merge(
    drivers[['driverId','surname','nationality']], on='driverId', how='left'
).merge(
    circuits[['circuitId','name','country','location','lat','lng']], on='circuitId', how='left'
).merge(
    constructors[['constructorId','constructor']], on='constructorId', how='left'
).merge(
    qualifying[['raceId','driverId','position']], on=['raceId','driverId'], how='left', suffixes=("","_grid")
)
f1.rename(columns={'name_x':'race_name','name_y':'circuit_name'}, inplace=True)

In [167]:
app = JupyterDash(__name__, external_stylesheets=[dbc.themes.CYBORG])


c:\Users\Marwan\AppData\Local\Programs\Python\Python313\Lib\site-packages\dash\dash.py:587: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.



In [168]:
df.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,date,circuitId,surname,nationality,circuit_name,country,location,lat,lng,position_grid
0,1,18,1,1,22,1,1.0,1,1,10.0,...,2008-03-16,1,Hamilton,British,Albert Park Grand Prix Circuit,Australia,Melbourne,-37.8497,144.968,1.0
1,2,18,2,2,3,5,2.0,2,2,8.0,...,2008-03-16,1,Heidfeld,German,Albert Park Grand Prix Circuit,Australia,Melbourne,-37.8497,144.968,5.0
2,3,18,3,3,7,7,3.0,3,3,6.0,...,2008-03-16,1,Rosberg,German,Albert Park Grand Prix Circuit,Australia,Melbourne,-37.8497,144.968,7.0
3,4,18,4,4,5,11,4.0,4,4,5.0,...,2008-03-16,1,Alonso,Spanish,Albert Park Grand Prix Circuit,Australia,Melbourne,-37.8497,144.968,12.0
4,5,18,5,1,23,3,5.0,5,5,4.0,...,2008-03-16,1,Kovalainen,Finnish,Albert Park Grand Prix Circuit,Australia,Melbourne,-37.8497,144.968,3.0


# EDA

In [169]:
def top_pilotes(df):
    top = df.groupby('surname')['points'].sum().sort_values(ascending=False).head(10).reset_index()
    return px.bar(top, x='points', y='surname', orientation='h',
                  color='points', color_continuous_scale='reds',
                  title='Top 10 pilotes (total points)',
                  template='plotly_dark')

def top_ecuries(df):
    top = df.groupby('constructor')['points'].sum().sort_values(ascending=False).head(10).reset_index()
    return px.bar(top, x='points', y='constructor', orientation='h',
                  color='points', color_continuous_scale='blues',
                  title='Top 10 écuries (total points)',
                  template='plotly_dark')

def map_circuits():
    circuits_unique = f1[['circuit_name','country','lat','lng']].drop_duplicates()
    return px.scatter_geo(circuits_unique,
                          lat='lat', lon='lng', text='circuit_name',
                          hover_name='country',
                          template='plotly_dark',
                          title='Carte des circuits de F1 (1950-2020)')

def grille_vs_points(df):
    df_corr = df.dropna(subset=['position', 'points'])
    df_corr['position'] = pd.to_numeric(df_corr['position'], errors='coerce')
    return px.scatter(df_corr, x='position', y='points',
                     title='Corrélation : position de grille vs points',
                     color='year', size='points', template='plotly_dark')

def evolution_gp():
    yearly = f1.groupby('year').agg({"raceId":"nunique"}).reset_index()
    return px.line(yearly, x='year', y='raceId', markers=True,
                   template='plotly_dark', title='Évolution du nombre de Grands Prix par an')

def heatmap_podium():
    podium = f1[f1['positionOrder'].isin([1, 2, 3])]
    heatmap_data = podium.groupby(['year', 'surname']).size().unstack(fill_value=0)
    fig = go.Figure(data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale='YlOrRd'))
    fig.update_layout(title='Heatmap des podiums par saison', template='plotly_dark')
    return fig

def kpis(df):
    total_races = df['raceId'].nunique()
    total_drivers = df['driverId'].nunique()
    total_teams = df['constructorId'].nunique()
    total_points = df['points'].sum()

    return dbc.Row([
        dbc.Col(dbc.Card([html.H4("Courses"), html.H2(f"{total_races}")], className="text-center p-3 bg-primary text-white rounded-3 shadow")),
        dbc.Col(dbc.Card([html.H4("Pilotes"), html.H2(f"{total_drivers}")], className="text-center p-3 bg-danger text-white rounded-3 shadow")),
        dbc.Col(dbc.Card([html.H4("Écuries"), html.H2(f"{total_teams}")], className="text-center p-3 bg-warning text-dark rounded-3 shadow")),
        dbc.Col(dbc.Card([html.H4("Points cumulés"), html.H2(f"{int(total_points)}")], className="text-center p-3 bg-success text-white rounded-3 shadow")),
    ], className="mb-4")

In [ ]:
app.layout = dbc.Container([
    html.H1("🏎️ Dashboard F1 — 1950 à 2020", className="text-center my-4"),
    dcc.Slider(id='annee-slider', min=f1['year'].min(), max=f1['year'].max(),
               step=1, value=f1['year'].max(),
               marks={int(y): str(y) for y in range(f1['year'].min(), f1['year'].max()+1, 10)},
               tooltip={"placement": "bottom", "always_visible": True}),
    html.Br(),
    html.Div(id='kpi-container'),
    html.Div(id='tabs-container', children=[])
], fluid=True)

@callback(
    Output('kpi-container', 'children'),
    Output('tabs-container', 'children'),
    Input('annee-slider', 'value')
)
def update_dashboard(selected_year):
    filtered = f1[f1['year'] <= selected_year]

    return kpis(filtered), html.Div([
        dcc.Tabs([
            dcc.Tab(label='Vue Globale', children=[
                dbc.Row([
                    dbc.Col(dcc.Graph(figure=top_pilotes(filtered)), width=6),
                    dbc.Col(dcc.Graph(figure=top_ecuries(filtered)), width=6)
                ]),
                dbc.Row([
                    dbc.Col(dcc.Graph(figure=evolution_gp()), width=12)
                ])
            ]),
            dcc.Tab(label='Carte des circuits', children=[
                dcc.Graph(figure=map_circuits())
            ]),
            dcc.Tab(label='Analyse grille vs points', children=[
                dcc.Graph(figure=grille_vs_points(filtered))
            ]),
            dcc.Tab(label='Heatmap des podiums', children=[
                dcc.Graph(figure=heatmap_podium())
            ])
        ])
    ])


# === Lancer l'app ===
app.run(mode='inline')

